# V118 - Exact assessment-twin transfer

Pre-registered plan: the audit found 209 test users with assessment vectors identical to train users, never used as signal. Twins share stated skills exactly, so the train twin mean labels predict the test twin. OOF version: train users with in-train exact twins get their twins mean, everyone else keeps the word-only baseline. Same grouped folds under a fresh seed, same locks. Deterministic, no training, no leakage across folds because twins are matched within training folds only.

In [1]:
import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 4747
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v118")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Twin means inside training folds, then gate

Twins match on decoded assessment vectors. Matches use fold-train users only, so validation labels never transfer.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
key = [tuple(num(blobs[u].get(q)) for q in questions) for u in list(train["user_id"])]
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()

by_raw = chats.groupby("user_id")["user_chat_text"].apply(list)
firsts = [str(by_raw.get(u, [""])[0]) if by_raw.get(u, []) else "NO_CHAT" for u in list(train["user_id"])]
fold_groups = np.array(firsts)
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_w, oof_t, fids = np.zeros_like(Y), np.zeros_like(Y), np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[tr], Y[tr, j])
        oof_w[va, j] = m.predict(W[va])
    pool = defaultdict(list)
    for u in tr:
        pool[key[u]].append(u)
    for u in va:
        mate = [v for v in pool[key[u]] if v != u]
        oof_t[u] = Y[mate].mean(0) if mate else oof_w[u]
    fids[va] = fi + 1
covered = sum(1 for u in range(n_train) if True)
base = ndcg5(Y, oof_w)
d = ndcg5(Y, oof_t) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_t[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"word={base.mean():.5f} twin={ndcg5(Y, oof_t).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT twin transfer" if (lo > GATE_FLOOR and wf >= 0) else "KEEP baseline; identical skills do not imply identical needs")

word=0.64156 twin=0.56553 delta=-0.07603 CI95=[-0.08154,-0.07053] worst-fold=-0.09210
KEEP baseline; identical skills do not imply identical needs
